In [12]:
%load_ext autoreload
%autoreload 2

from radar.components import geometry
from radar.utils.calculate import convert
from radar.utils.typing.enums import FrequencyUnit
from radar.utils.typing.units import Frequency

from radar.components import Element
from radar.utils.calculate import pattern
from radar.utils.typing import (
    PhaseUnit,
    DirectionDomain,
    FigureType,
    AmplitudeDomain,
    Angle,
    AmplitudeUnit,
)

from radar.components.array import Array
import polars as pl

from radar.utils.typing.constants import DataHeader

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
# 1. Initialize your population
from radar.optimiser.biology import Population

pop = Population(
    5,
    10,
    [DataHeader.X_POS_M, DataHeader.Y_POS_M, DataHeader.GEOM_AMP_GAIN_DB],
    25,
    [-0.5, -0.5, -3],
    [0.5, 0.5, 0],
    0.60,
)
if "generation" in locals():
    del generation

FigureWidget({
    'data': [],
    'layout': {'legend': {'itemclick': 'toggle',
                          'itemdoubleclick': 'toggleothers',
                          'orientation': 'h',
                          'x': 1,
                          'xanchor': 'right',
                          'y': 1.02,
                          'yanchor': 'bottom'},
               'showlegend': True,
               'template': '...',
               'title': {'text': 'Live Dynamic Plot'}}
})

In [14]:
best = pop._elite_community._organisms[0]
best.save("tmp5.csv")

In [15]:
pop.seed(Organism.load("tmp5.csv", [-0.5, -0.5, -10], [0.5, 0.5, 0]))

In [ ]:

from  import fitness_function

import datetime

import logging

# Mute the kaleido logger specifically
logging.getLogger("kaleido").setLevel(logging.ERROR)
# # 2. Define how many generations you want to evolve
num_epoch = 500000

print("Starting optimization loototal_diffp...\n")

start = 1
if "epoch" in locals():
    start = epoch

for epoch in range(start, num_epoch + 1):
    # 3. Propagate the population to the next generation
    # Pass the function name without parentheses!


    pop.propagate_epochs(
        fitness_fn=fitness_function, generations_per_epoch=5, current_epoch=epoch
    )

    best = pop._elite_community._organisms[0]
    best.save("tmp7.csv")
        # tmp = pop._elite_community._organisms[0]
        # calculate_genetic_health3(tmp, generation, True)

    # print(f"Generation {generation}/{num_generations}:")

ModuleNotFoundError: No module named 'fitness'

In [44]:
x = pop._elite_community._organisms[0]._chromosomes[0]
y = pop._elite_community._organisms[0]._chromosomes[1]
c = pop._elite_community._organisms[0]._chromosomes[2]

az_bound = 90
el_bound = 90
az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


element_pattern = pattern.Isotropic()
freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

cf = Frequency(1, FrequencyUnit.GIGAHERTZ)

array_geometry = geometry.CustomGeometry(
    x.df.to_numpy().ravel(), y.df.to_numpy().ravel()
)
df = array_geometry.df
updated_df = df.with_columns(c.df.to_series().alias(DataHeader.GEOM_AMP_GAIN_DB)).drop(
    DataHeader.GEOM_AMP_GAIN_LIN
)
array_geometry.gains = updated_df

arr = Array(antenna_element, array_geometry)

arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(1, FrequencyUnit.GIGAHERTZ),
)

arr.plot.geometry()